# 项目图片生成示例

这个 notebook 用来演示项目里推荐的文生图调用方式。

推荐的两种路径：
- Python 内部直接调用：`generate_image(...)`
- 前端/联调调用：`POST http://127.0.0.1:8787/api/t2i`

以后如果你在项目的 Python 代码里生成图片，优先使用第一种方式。


In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    # 从当前目录开始逐级向上查找，直到找到包含 Code 目录的项目根目录。
    for candidate in [start, *start.parents]:
        if (candidate / 'Code').exists():
            return candidate
    raise RuntimeError(f'无法从当前路径定位仓库根目录：{start}')


ROOT = find_repo_root(Path.cwd().resolve())
CODE_DIR = (ROOT / 'Code').resolve()
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

# notebook 额外输出目录，只用于保存测试时的附加文件。
TEST_OUTPUT_DIR = ROOT / 'outputs' / 'notebook_tests'
TEST_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('项目根目录 =', ROOT)
print('代码目录 =', CODE_DIR)
print('notebook 输出目录 =', TEST_OUTPUT_DIR)


In [ ]:
from image_model.function_ImageGeneration import generate_image, get_selected_image_model_code, get_selected_image_model_config

# 检查当前项目到底会选中哪个图片模型配置。
model_code = get_selected_image_model_code()
model_config = get_selected_image_model_config()
base_url, model_name = model_config.resolve_endpoint(feature='text_to_image')

print('当前选中的图片模型 code =', model_code)
print('配置中的 code =', model_config.code)
print('实际调用的模型名 =', model_name)
print('base_url =', base_url)
print('API Key 环境变量名 =', model_config.api_key_env)
print('最大 prompt 长度 =', model_config.max_prompt_chars)


In [ ]:
# 下面这些是项目里文生图最常用的参数。
# PROMPT: 你想生成什么画面。
# TRIGGER: 这次请求属于哪种触发类型，会影响最终 prompt 的包装方式。
# THEME: 视觉主题，会影响项目内置风格描述。
# SCENE_ID: 场景标识，用于缓存；相同场景可以复用已生成图片。
# CHARACTERS: 角色一致性信息，可传入角色名和外观描述。

PROMPT = 'TRPG desert ruin at dusk, broken stone gate, blowing sand, cinematic composition, no text'
TRIGGER = 'manual'
THEME = 'neon'
SCENE_ID = 'notebook-image-model-test'
CHARACTERS = []

# 如果你要测试前端那种 HTTP 调用方式，就把 USE_HTTP_SERVER 改成 True。
# 只有在本地服务已经启动时才需要打开它。
SERVER_ENDPOINT = 'http://127.0.0.1:8787/api/t2i'
USE_HTTP_SERVER = False

PROMPT


## 1. 项目内推荐用法：直接调用 `generate_image(...)`

这就是以后在项目 Python 代码里最推荐的生成图片方式。

它已经帮你处理好了：
- 读取 `image_model` 用户偏好
- 读取 `data_ImageModel.yml`
- 自动解析 API Key
- 组合 TRPG 项目里的最终 prompt
- 复用缓存图片
- 把图片保存到 `outputs/t2i`


In [ ]:
from IPython.display import Image, display

# 这是项目内部最推荐的调用方式。
result = generate_image(
    prompt=PROMPT,
    trigger=TRIGGER,
    theme=THEME,
    scene_id=SCENE_ID,
    characters=CHARACTERS,
    save_output=True,
    use_cache=True,
)

print('提供方 =', result.provider)
print('模型 code =', result.model_code)
print('模型名 =', result.model_name)
print('是否命中缓存 =', result.cached)
print('最终 prompt =', result.prompt)
print('输出文件路径 =', result.output_path)
print('图片 MIME 类型 =', result.mime_type)

if result.output_path:
    display(Image(filename=str(result.output_path)))


## 1.1 参考图生成示例

如果你想让 Gemini 参考已有图片继续生成，可以传入 `reference_images`。

目前支持的参考图输入格式：
- 本地文件路径字符串
- `Path(...)`
- `{"path": "..."}`
- `{"data_url": "data:image/png;base64,..."}`
- `{"bytes_base64": "...", "mime_type": "image/png"}`


In [ ]:
# 改成你自己的参考图路径后，再执行这一格。
REFERENCE_IMAGE_PATH = None

if not REFERENCE_IMAGE_PATH:
    print('已跳过参考图示例。把 REFERENCE_IMAGE_PATH 改成实际图片路径后再运行。')
else:
    reference_result = generate_image(
        prompt='在保留参考图主体特征的前提下，转成赛博霓虹风格，增加发光装置与未来都市背景',
        trigger='manual',
        theme='neon',
        scene_id='notebook-reference-image-test',
        reference_images=[REFERENCE_IMAGE_PATH],
        save_output=True,
        use_cache=True,
    )

    print('参考图数量 =', reference_result.reference_count)
    print('最终 prompt =', reference_result.prompt)
    print('输出文件路径 =', reference_result.output_path)

    if reference_result.output_path:
        display(Image(filename=str(reference_result.output_path)))


## 2. 前后端联调用法：请求本地 HTTP 接口

这部分对应前端实际调用图片服务的方式。

先在项目根目录启动本地服务：

`python Code/image_model/function_ImageGeneration.py`

然后把 `USE_HTTP_SERVER = True`，再执行下面这一格。


In [ ]:
import requests

if not USE_HTTP_SERVER:
    print('已跳过 HTTP 联调测试。需要时把 USE_HTTP_SERVER 改成 True。')
else:
    response = requests.post(
        SERVER_ENDPOINT,
        json={
            'prompt': PROMPT,
            'trigger': TRIGGER,
            'theme': THEME,
            'scene_id': SCENE_ID,
            'characters': CHARACTERS,
            'reference_images': [],
        },
        timeout=120,
    )
    response.raise_for_status()
    data = response.json()
    print(data)
    if data.get('image_url'):
        display(Image(url=data['image_url']))
